In [1]:
# Ejecutar bootstrap() para (re)crear ./data y ./products
from amazon_bot.services.bootstrap import bootstrap
bootstrap()

# Importar la clase del bot 
from amazon_bot.bot.amazon_bot import AmazonBot

# Instanciar el bot
bot1 = AmazonBot()

In [2]:
# === Chat (bot vs usuario)  ===
from IPython.display import display, HTML
import ipywidgets as W
from datetime import datetime
import html

# ---------- Estilos ----------
display(HTML("""
<style> 
:root {
  --bot-bg: #f1f3f4;
  --bot-fg: #202124;
  --user-bg: #0b5394;
  --user-fg: #ffffff;
  --card-bg: #ffffff;
  --border: #e5e7eb;
}

.chat-card {
  max-width: 860px;
  border: 1px solid var(--border);
  border-radius: 14px;
  background: var(--card-bg);
  font-family: system-ui, -apple-system, Segoe UI, Roboto, Helvetica, Arial, sans-serif;
  overflow: hidden;
}
.chat-header {
  display:flex; align-items:center; gap:12px;
  padding:10px 14px; border-bottom:1px solid var(--border); background:#8abcee;
}
.chat-header .avatar {
  width:36px; height:36px; border-radius:50%;
  display:grid; place-items:center; color:white; font-weight:700; background:#0b5394;
}
.chat-header .title { font-weight:700; }
.chat-header .status { color:#5f6368; font-size:12px; }

.messages {
  height: 430px; overflow-y: auto; padding:14px;
  background:white;
}

.msg-row { display:flex; margin:8px 0; }
.msg-row.bot   { justify-content:flex-start; }
.msg-row.user  { justify-content:flex-end; }

.bubble {
  max-width: 72%;
  padding:10px 12px; border-radius:14px; line-height:1.35; font-size:14.5px;
  box-shadow: 0 1px 1px rgba(0,0,0,.03);
  white-space: pre-wrap; word-wrap: break-word;
}
.bubble.bot  { background:var(--bot-bg);  color:var(--bot-fg);  border-bottom-left-radius:6px; }
.bubble.user { background:var(--user-bg); color:var(--user-fg); border-bottom-right-radius:6px; }

.meta { font-size:11px; color:#6b7280; margin-top:4px; }

.input-bar {
  display:flex; gap:8px; padding:10px; border-top:1px solid var(--border); background:#fafafa;
}
.qa-btn button { 
  border:1px solid var(--border); border-radius:18px; padding:4px 10px; background:white;
  font-size:13px; cursor:pointer;
}
.qa-btn button:hover { background:#f6f8fa; }
</style>
"""))

# ---------- Estado ----------
bot = AmazonBot()
messages = []  # [{'role':'bot'|'user', 'text':str, 'ts': 'HH:MM'}]

def now_str(): 
    return datetime.now().strftime("%H:%M")

def escape(s):
    return html.escape(str(s))

# ---------- Widgets ----------
chat_html = W.HTML(layout=W.Layout(border="1px solid #ddd", padding="6px"))
inp = W.Textarea(placeholder="Escribe aquí y presiona Enter…", rows=1, layout=W.Layout(flex="1"))
btn_send = W.Button(description="Enviar", button_style="warning", layout=W.Layout(width="110px"))
btn_reset = W.Button(description="Reiniciar", layout=W.Layout(width="110px"))

# Contenedores
chat_box = W.VBox([chat_html])
input_box = W.HBox([inp, btn_send, btn_reset])
ui_box = W.VBox([chat_box, input_box])

display(ui_box)


# ---------- Funciones ----------
def render():
    html_msgs = []
    for m in messages:
        role = "bot" if m["role"] == "bot" else "user"
        html_msgs.append(
            f"<div class='msg-row {role}'>"
            f"<div class='bubble {role}'>{escape(m['text'])}</div>"
            f"</div>"
        )
    chat_html.value = f"""
<div class='chat-card'>
  <div class='chat-header'>
    <div class='avatar'>A</div>
    <div>
      <div class='title'>Asistente de AMAZON</div>
    </div>
  </div>

  <div class='messages'>
    {''.join(html_msgs)}
  </div>
</div>
"""

def add_message(role, text):
    messages.append({"role": role, "text": text})
    render()

def send_message(_=None):
    msg = inp.value.strip()
    if not msg:
        return
    inp.value = ""
    add_message("user", msg)
    try:
        resp = bot.handle(msg)
    except Exception as e:
        resp = f"Ocurrió un error: {e}"
    add_message("bot", resp)
    if getattr(bot, "ctx", None) and getattr(bot.ctx, "state", "") == "FIN":
        inp.disabled = True
        btn_send.disabled = True

def send_from_chip(b):
    inp.value = b.description
    send_message()

def reset_chat(_=None):
    global bot, messages
    bot = AmazonBot()
    messages = []
    inp.disabled = False
    btn_send.disabled = False
    add_message("bot", bot.prompt())

# ---------- Eventos ----------
btn_send.on_click(send_message)
btn_reset.on_click(reset_chat)

# Captura Enter en Textarea
def on_text_change(change):
    if change["name"] == "value" and change["type"] == "change":
        val = change["new"]
        if val.endswith("\n"):
            inp.value = val.rstrip("\n")
            send_message()
inp.observe(on_text_change, names="value")

# ---------- Iniciar ----------
add_message("bot", bot.prompt())
